#### Master INNER JOIN,LEFT JOIN,GROUP BY,HAVING,subqueries and set operators.

*Goal:* Solve business reporting and data transformations problems

* Topics to master

1)Joins,duplicate rows,anti-joins,self-joins

2)Aggregations,GROUP BY,HAVING,conditional aggregation

3)subqueries,CTEs,correlated subqueries

### Practice problems grouped by topic (with short hints)

Topic 1 — Joins, duplicate rows, anti-joins, self-joins 1.1 Inner join: List order_id, order_date, customer name, city, and sales rep full name for all orders that have a customer and a sales rep.

Hint: JOIN orders -> customers and orders -> employees (sales rep).
1.2 LEFT JOIN: Show every customer and their most recent order_date (NULL if never ordered).

Hint: LEFT JOIN and aggregate MAX(order_date) grouped by customer.
1.3 Duplicate rows from joins: When you join orders to order_items you get multiple rows per order. Produce one row per order with total_items (sum of quantity) and computed_total (SUM(quantity * unit_price)). Ensure results match orders.total_amount.

Hint: GROUP BY order_id on order_items; compare with orders.total_amount.
1.4 Anti-join: Find customers who have never placed an order.

Hint: LEFT JOIN orders ON customer_id IS NULL OR use NOT EXISTS.
1.5 Anti-join variant: Find orders with no associated customer (guest orders).

Hint: WHERE customer_id IS NULL or LEFT JOIN customers WHERE customers.customer_id IS NULL.
1.6 Self-join (manager/subordinate): List each sales associate and their manager (full names). Include associates that have no manager (show NULL manager).

Hint: JOIN employees e ON e.manager_id = m.employee_id.
1.7 Self-join peers: For a given manager, list pairs of associates who report to same manager (use caution to avoid duplicate pairs).

Hint: Self-join employees to itself and filter e1.employee_id < e2.employee_id.
1.8 Deduplication: Identify customer names that appear more than once (e.g., 'Acme Corp' in multiple cities).

Hint: GROUP BY name HAVING COUNT(*) > 1.
1.9 Set operators: Produce three result sets (use UNION / INTERSECT / EXCEPT):

Customers who ordered in August 2024,
Customers who ordered in September 2024,
Use INTERSECT to find customers who ordered in both months; EXCEPT to find those who ordered in August but not September.
Hint: Use SELECT DISTINCT customer_id FROM orders WHERE order_date BETWEEN ... ; then set operators.
Topic 2 — Aggregations, GROUP BY, HAVING, conditional aggregation 2.1 Revenue per customer: Show customer_id, customer name, total_revenue ordered desc.

Hint: JOIN orders -> customers and SUM(orders.total_amount) GROUP BY customer.
2.2 Monthly revenue: For each year-month (e.g., 2024-08), show total revenue. Only show months with revenue > 1000.

Hint: Use DATE_TRUNC or EXTRACT/YEAR+MONTH concatenation and HAVING SUM(...) > 1000.
2.3 Conditional aggregation: For each customer show total_revenue and breakdown by category (Hardware, Electronics, Service) in separate columns.

Hint: JOIN orders -> order_items -> products and SUM(CASE WHEN category='X' THEN quantity * unit_price ELSE 0 END).
2.4 Top product per category: For each category find the product with highest revenue and its revenue.

Hint: Use GROUP BY product, then a subquery/CTE to pick max per category (or window function if allowed).
2.5 Orders with unique product counts: Show order_id and number_of_distinct_products.

Hint: COUNT(DISTINCT product_id) from order_items GROUP BY order_id.
2.6 Percent of company revenue by customer: Show customer, total_revenue and pct_of_total (two decimals).

Hint: Compute company_total via subquery or window SUM() and divide.
2.7 HAVING usage: Show customers with at least 2 orders and total_revenue > 1500.

Hint: GROUP BY customer_id HAVING COUNT(order_id) >= 2 AND SUM(total_amount) > 1500.
Topic 3 — Subqueries, CTEs, correlated subqueries 3.1 Scalar subquery: In one query show each customer and a column company_total_revenue (same constant per row).

Hint: SELECT ..., (SELECT SUM(total_amount) FROM orders) AS company_total.
3.2 Correlated subquery: For each order, show order_id, total_amount, and the customer's average order amount (computed per row using correlated subquery).

Hint: SELECT o.*, (SELECT AVG(total_amount) FROM orders o2 WHERE o2.customer_id = o.customer_id) AS cust_avg FROM orders o.
3.3 CTE for readability: Build a CTE customer_revenue (customer_id, total_revenue) then select top 5 customers by revenue.

Hint: WITH customer_revenue AS ( ... ) SELECT * FROM customer_revenue ORDER BY total_revenue DESC LIMIT 5.
3.4 EXISTS vs JOIN: List customers who have at least one shipped order using EXISTS. Then rewrite with INNER JOIN and discuss differences in duplicates and performance.

Hint: EXISTS returns one row per customer regardless of number of matching orders.
3.5 Correlated "top N per group" using subquery: For each category, list products that are within the top 2 by revenue in that category.

Hint: Use correlated subquery counting how many products in same category have higher revenue <= 1, or use window functions (RANK/DENSE_RANK).
3.6 NOT EXISTS anti-join: Show customers who have no shipped orders (prefer NOT EXISTS over LEFT JOIN ... IS NULL).

Hint: WHERE NOT EXISTS (SELECT 1 FROM orders o WHERE o.customer_id = c.customer_id AND o.status='shipped').
Combined business reporting challenges C1) Customer lifetime report: - For each customer, show: customer_id, name, first_order_date, last_order_date, total_orders, total_revenue, avg_order_value, days_since_last_order (relative to a given date e.g., '2024-09-15'). - Hint: Use GROUP BY with MIN/MAX, COUNT, SUM, AVG and a reference date for DATEDIFF.

C2) Sales rep performance dashboard: - For each sales rep show: sales_rep_id, full_name, total_orders_handled, revenue_handled, avg_order_value, num_customers_handled (distinct customers), pct_of_team_revenue (team = all reps under same manager). - Hint: Combine joins, GROUP BY, COUNT(DISTINCT), and a window function or subquery to get team totals (or use a CTE grouping by manager).

C3) Identify potential data issues: - Find orders where orders.total_amount differs from the sum of order_items (quantity * unit_price). Show order_id, orders.total_amount, items_total, difference. - Hint: LEFT JOIN aggregate of order_items and compare.

C4) Historical churn detection (moderate): - Define churn as customers who had orders in the prior 90-day window but none in the most recent 30-day window. Use orders data and date ranges to classify customers as 'Active' / 'ChurnRisk' / 'Churned'. - Hint: Use conditional aggregation on date ranges (SUM(CASE WHEN order_date BETWEEN ... THEN 1 ELSE 0 END) per customer).

C5) De-duplicate and merge: - You see two customer rows for 'Acme Corp' (different customer_id). Produce a canonicalization plan: list groups of likely duplicates (same name different city), and compute combined revenue per duplicate-group to inform merging. - Hint: GROUP BY name HAVING COUNT(*) > 1; then aggregate orders by name to produce combined metrics.

Hints & best practices

Use explicit column lists in SELECT.
Prefer NOT EXISTS for anti-joins (clear semantics with NULLs) or LEFT JOIN ... IS NULL as an alternative.
When joining many-to-many data, aggregate before joining if you need one row per left-side entity (pre-aggregate order_items to order-level totals).
Use CTEs to break complex queries into readable named steps.
Correlated subqueries are expressive but can be slow at scale — consider converting to JOINs or CTEs with aggregates when performance matters.
Validate joins: to detect accidental row multiplication, compare counts before/after or use COUNT(DISTINCT primary_key).


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS sql.intermed;

-- Customers
CREATE TABLE IF NOT EXISTS sql.intermed.customers (
  customer_id INT PRIMARY KEY,
  name VARCHAR(100) NOT NULL,
  city VARCHAR(50),
  created_date DATE NOT NULL
);

INSERT INTO sql.intermed.customers (customer_id, name, city, created_date) VALUES
(1, 'Acme Corp', 'New York', '2023-01-15'),
(2, 'Beta LLC', 'Chicago', '2023-02-20'),
(3, 'Acme Corp', 'Boston', '2024-03-05'), -- duplicate name (different location)
(4, 'Delta Co', NULL, '2022-11-01'),      -- NULL city
(5, 'Echo Ltd', 'Seattle', '2024-07-10'),
(6, 'Foxtrot Inc', 'Chicago', '2024-08-02'),
(7, 'Guest', NULL, '2024-09-01'),         -- guest-like customer for testing
(8, 'Gamma Inc', 'Austin', '2024-08-15');

-- Employees (sales reps) - includes manager relationships for self-join practice
CREATE TABLE IF NOT EXISTS sql.intermed.employees (
  employee_id INT PRIMARY KEY,
  first_name VARCHAR(30),
  last_name VARCHAR(30),
  manager_id INT,  -- self reference
  hire_date DATE,
  role VARCHAR(50)
);

INSERT INTO sql.intermed.employees (employee_id, first_name, last_name, manager_id, hire_date, role) VALUES
(101, 'Priya', 'Shah', NULL, '2019-05-01', 'Sales Director'),
(102, 'Liam', 'Olsen', 101, '2020-02-10', 'Account Manager'),
(103, 'Maya', 'Singh', 101, '2021-06-15', 'Account Manager'),
(104, 'Noah', 'Park', 102, '2022-01-20', 'Sales Associate'),
(105, 'Zara', 'Khan', 102, '2023-09-05', 'Sales Associate'),
(106, 'Omar', 'Diaz', 103, '2024-02-12', 'Sales Associate');

-- Products
CREATE TABLE IF NOT EXISTS sql.intermed.products (
  product_id INT PRIMARY KEY,
  name VARCHAR(100),
  category VARCHAR(50),
  price DECIMAL(10,2)
);

INSERT INTO sql.intermed.products (product_id, name, category, price) VALUES
(1001, 'Widget A', 'Hardware', 1250.00),
(1002, 'Widget B', 'Hardware', 450.00),
(1003, 'Gadget X', 'Electronics', 780.50),
(1004, 'Service Plan', 'Service', 300.00),
(1005, 'Addon Y', 'Electronics', 99.99);

-- Orders: note some orders have customer_id NULL (guest), some have NULL sales_rep_id
CREATE TABLE IF NOT EXISTS sql.intermed.orders (
  order_id INT PRIMARY KEY,
  customer_id INT,
  order_date DATE NOT NULL,
  sales_rep_id INT,
  status VARCHAR(20),       -- 'shipped','pending','canceled'
  total_amount DECIMAL(10,2) NOT NULL
);

INSERT INTO sql.intermed.orders (order_id, customer_id, order_date, sales_rep_id, status, total_amount) VALUES
(2001, 1, '2024-08-30', 102, 'shipped', 1250.00),
(2002, 2, '2024-09-01', 103, 'pending', 780.50),
(2003, 1, '2024-09-03', 102, 'shipped', 2050.00),
(2004, NULL, '2024-09-05', NULL, 'pending', 450.00),   -- guest order (no customer)
(2005, 3, '2024-08-28', 103, 'shipped', 1250.00),
(2006, 4, '2024-07-15', 101, 'canceled', 300.00),
(2007, 5, '2024-09-10', 105, 'shipped', 879.49),      -- multiple product order total
(2008, 6, '2024-08-05', 104, 'shipped', 450.00),
(2009, 8, '2024-09-12', 106, 'pending', 550.00),
(2010, 2, '2024-09-12', 103, 'shipped', 99.99);

-- Order items: demonstrates one-to-many and duplicate products in the same order
CREATE TABLE IF NOT EXISTS sql.intermed.order_items (
  order_item_id INT PRIMARY KEY,
  order_id INT NOT NULL,
  product_id INT NOT NULL,
  quantity INT NOT NULL,
  unit_price DECIMAL(10,2) NOT NULL
);

INSERT INTO sql.intermed.order_items (order_item_id, order_id, product_id, quantity, unit_price) VALUES
(1, 2001, 1001, 1, 1250.00),
(2, 2002, 1003, 1, 780.50),
(3, 2003, 1001, 1, 1250.00),
(4, 2003, 1002, 2, 450.00),  -- order 2003 has 2 items => join duplicates
(5, 2004, 1002, 1, 450.00),
(6, 2005, 1001, 1, 1250.00),
(7, 2007, 1003, 1, 780.50),
(8, 2007, 1005, 1, 99.99),
(9, 2008, 1002, 1, 450.00),
(10, 2009, 1002, 1, 450.00),
(11, 2010, 1005, 1, 99.99),
(12, 2003, 1005, 1, 99.99);  -- same order 2003 includes three items, one product repeated across different rows


In [0]:
%sql
SHOW TABLES IN sql.intermed;

In [0]:
%sql
--Inner join: List order_id, order_date, customer name, city, and sales rep full name for all orders that have a customer and a sales rep.
SELECT o.order_id,o.order_date,c.name,c.city from sql.intermed.orders o INNER JOIN
sql.intermed.customers c ON o.customer_id=c.customer_id INNER JOIN
sql.intermed.employees e ON o.sales_rep_id=e.employee_id;

In [0]:
%sql
--1.2 LEFT JOIN: Show every customer and their most recent order_date (NULL if never ordered).
SELECT c.name,max(o.order_date) as latest_order_date FROM sql.intermed.customers c LEFT JOIN
sql.intermed.orders o ON c.customer_id = o.customer_id GROUP BY c.name;

In [0]:
%sql
--1.3 Duplicate rows from joins: When you join orders to order_items you get multiple rows per order. Produce one row per order with total_items (sum of quantity) and computed_total (SUM(quantity * unit_price)). Ensure results match orders.total_amount.
SELECT order_id, SUM(quantity) as total_items, SUM(quantity* unit_price) as computed_total FROM sql.intermed.order_items GROUP BY order_id

In [0]:
%sql
--Anti-join: Find customers who have never placed an order.
SELECT c.name FROM sql.intermed.customers c 
LEFT JOIN 
sql.intermed.orders o 
ON c.customer_id IS NULL;

In [0]:
%sql
--1.4 Anti-join: Find customers who have never placed an order.
SELECT c.customer_id, c.name
FROM sql.intermed.customers c
LEFT JOIN sql.intermed.orders o
  ON c.customer_id = o.customer_id
WHERE o.order_id IS NULL;

-- Alternative using NOT EXISTS (preferred for anti-joins):
-- SELECT c.customer_id, c.name
-- FROM sql.intermed.customers c
-- WHERE NOT EXISTS (
--   SELECT 1 FROM sql.intermed.orders o WHERE o.customer_id = c.customer_id
-- );

In [0]:
%sql
--1.5 Anti-join variant: Find orders with no associated customer (guest orders).
SELECT o.order_id,o.order_date,c.name FROM sql.intermed.orders o
LEFT JOIN
sql.intermed.customers c ON o.customer_id=c.customer_id
WHERE c.customer_id IS NULL;

In [0]:
%sql
--1.6 Self-join (manager/subordinate): List each sales associate and their manager (full names). Include associates that have no manager (show NULL manager).
SELECT CONCAT(e.first_name,' ',e.last_name) as employee_name,m.first_name as manager FROM sql.intermed.employees e LEFT JOIN sql.intermed.employees m ON e.employee_id = m.manager_id where m.manager_id IS NULL;

In [0]:
%sql
--1.7 Self-join peers: List pairs of associates who report to the same manager,
--avoiding duplicate/reversed pairs by requiring e1.employee_id < e2.employee_id.
SELECT m.first_name || ' ' || m.last_name AS manager_name,
       CONCAT(e1.first_name,' ',e1.last_name) AS associate_1,
       CONCAT(e2.first_name,' ',e2.last_name) AS associate_2
FROM sql.intermed.employees e1
JOIN sql.intermed.employees e2
  ON e1.manager_id = e2.manager_id          -- same manager
 AND e1.employee_id < e2.employee_id        -- prevents (a,b) AND (b,a) duplicates, also excludes self-pairs
JOIN sql.intermed.employees m
  ON e1.manager_id = m.employee_id;         -- resolve manager name

-- Without the e1.employee_id < e2.employee_id condition, each pair would
-- appear twice (A-B and B-A); using '<' keeps only one ordered pair per duo.

In [0]:
%sql
--1.8 Deduplication: Identify customer names that appear more than once (e.g., 'Acme Corp' in multiple cities).
SELECT name,count(*) as total_count FROM sql.intermed.customers GROUP BY name having total_count>1;

#### Topic 2 — Aggregations, GROUP BY, HAVING, conditional aggregation 

In [0]:
%sql
--2.1 Revenue per customer: Show customer_id, customer name, total_revenue ordered desc.
SELECT c.customer_id,c.name,SUM(o.total_amount) as total_revenue FROM sql.intermed.customers c 
INNER JOIN
sql.intermed.orders o ON c.customer_id = o.customer_id
GROUP BY c.customer_id,c.name ORDER BY total_revenue DESC;

In [0]:
%sql
--2.2 Monthly revenue: For each year-month (e.g., 2024-08), show total revenue. Only show months with revenue > 1000.
SELECT date_format(order_date,'yyyy-MM') as year_month,
SUM(total_amount) as total_revenue
FROM sql.intermed.orders
GROUP BY year_month
HAVING total_revenue >1000;
-- Alternative using EXTRACT (YEAR ... ) || '-' || EXTRACT (MONTH ...):
-- SELECT CONCAT(YEAR(order_date), '-', LPAD(MONTH(order_date), 2, '0')) AS year_month, ...

In [0]:
%sql
SELECT c.name,
       SUM(oi.quantity * oi.unit_price) AS total_revenue,
       SUM(CASE WHEN p.category='Hardware' THEN oi.quantity*oi.unit_price ELSE 0 END) AS Hardware,
       SUM(CASE WHEN p.category='Electronics' THEN oi.quantity*oi.unit_price ELSE 0 END) AS Electronics,
       SUM(CASE WHEN p.category='Service' THEN oi.quantity*oi.unit_price ELSE 0 END) AS Service
FROM sql.intermed.customers c
INNER JOIN sql.intermed.orders o ON c.customer_id = o.customer_id
INNER JOIN sql.intermed.order_items oi ON o.order_id = oi.order_id
INNER JOIN sql.intermed.products p ON oi.product_id = p.product_id
GROUP BY c.customer_id, c.name;

In [0]:
%sql
--2.4 Top product per category: For each category find the product with highest revenue and its revenue.
WITH product_revenue AS (
  SELECT p.category,
         p.product_id,
         p.name AS product_name,
         SUM(oi.quantity * oi.unit_price) AS total_revenue
  FROM sql.intermed.products p
  INNER JOIN sql.intermed.order_items oi
    ON p.product_id = oi.product_id
  GROUP BY p.category, p.product_id, p.name
),
ranked AS (
  SELECT category,
         product_name,
         total_revenue,
         RANK() OVER (PARTITION BY category ORDER BY total_revenue DESC) AS rnk
  FROM product_revenue
)
SELECT category, product_name, total_revenue
FROM ranked
WHERE rnk = 1
ORDER BY total_revenue DESC;

-- Alternative without window functions (correlated subquery):
-- WITH product_revenue AS (
--   SELECT p.category, p.product_id, p.name AS product_name,
--          SUM(oi.quantity * oi.unit_price) AS total_revenue
--   FROM sql.intermed.products p
--   JOIN sql.intermed.order_items oi ON p.product_id = oi.product_id
--   GROUP BY p.category, p.product_id, p.name
-- )
-- SELECT pr.*
-- FROM product_revenue pr
-- WHERE pr.total_revenue = (
--   SELECT MAX(pr2.total_revenue)
--   FROM product_revenue pr2
--   WHERE pr2.category = pr.category
-- )
-- ORDER BY pr.total_revenue DESC;

In [0]:
%sql
--2.5 Orders with unique product counts: Show order_id and number_of_distinct_products.
SELECT order_id,
       COUNT(DISTINCT product_id) AS number_of_distinct_products
FROM sql.intermed.order_items
GROUP BY order_id
ORDER BY order_id;

In [0]:
%sql
SELECT order_id,COUNT(DISTINCT product_id) AS number_of_distinct_products FROM sql.intermed.order_items GROUP BY order_id ORDER BY order_id;

In [0]:
%sql
--2.6 Percent of company revenue by customer: Show customer, total_revenue and pct_of_total (two decimals).
SELECT c.customer_id,
       c.name,
       cr.total_revenue,
       ROUND(cr.total_revenue / (SELECT SUM(total_amount) FROM sql.intermed.orders) * 100, 2) AS pct_of_total
FROM (
  SELECT o.customer_id, SUM(o.total_amount) AS total_revenue
  FROM sql.intermed.orders o
  WHERE o.customer_id IS NOT NULL
  GROUP BY o.customer_id
) cr
INNER JOIN sql.intermed.customers c
  ON cr.customer_id = c.customer_id
ORDER BY cr.total_revenue DESC;

-- Alternative using window SUM() instead of a scalar subquery:
-- SELECT customer_id, name, total_revenue,
--        ROUND(total_revenue * 100 / SUM(total_revenue) OVER (), 2) AS pct_of_total
-- FROM (
--   SELECT c.customer_id, c.name, SUM(o.total_amount) AS total_revenue
--   FROM sql.intermed.orders o
--   INNER JOIN sql.intermed.customers c ON o.customer_id = c.customer_id
--   GROUP BY c.customer_id, c.name
-- )
-- ORDER BY total_revenue DESC;


In [0]:
%sql
--2.7 HAVING usage: Show customers with at least 2 orders and total_revenue > 1500.
SELECT c.name,count(o.order_id) as orders_count,sum(o.total_amount) as total_revenue 
FROM sql.intermed.customers c 
INNER JOIN
sql.intermed.orders o 
ON c.customer_id = o.customer_id
GROUP BY c.name
HAVING orders_count >=2 AND total_revenue>1500;